(course-core-combined-sources)=

# Module 3: Combined Sources (Trajectories)

```{admonition} Glossary: Combined Sources
:class: info
**Combined Sources** refers to MolSysMT's ability to treat a list of different data forms (e.g., a PDB file and an XTC trajectory) as a single, unified molecular system.
```

You have already seen us pass lists of files to `view()` or `has_attribute()`. This is a powerful feature of MolSysMT: it can merge multiple data sources into a single virtual system without any manual effort.

Traditionally, combining a **PDB** (topology) and an **XTC** (trajectory) required loading them into a specific object or using a complex merge script. In MolSysMT, you just provide them together in a list, and the framework resolves the system for you.

In [ ]:
import molsysmt as msm
from molsysmt import systems

topology_file = systems['chicken villin HP35']['chicken_villin_HP35_solvated.h5msm']
trajectory_file = systems['chicken villin HP35']['traj_chicken_villin_HP35_solvated.dcd']

### 1. The Virtual System
When you pass a list of complementary forms, MolSysMT can treat it as a single molecular system. It first verifies that no more than one item provides the topology and then checks that the available atom counts agree. A list containing two topology-providing items represents separate systems instead.

In [ ]:
composite_system = [topology_file, trajectory_file]

print(f"Is it a valid system? {msm.is_a_molecular_system(composite_system)}")
print(f"Final form: {msm.get_form(composite_system)}")

### 2. Rules of Precedence
If two files in your list contain the same attribute (e.g., both have coordinates), MolSysMT will use the data from the **first** file in the list. This allows you to "patch" or override information very easily.

In [ ]:
# Ask for the source of 'atom_name'
source = msm.where_is_attribute(composite_system, 'atom_name')
print(f"Atom names are coming from: {source}")

# Ask for the source of 'coordinates'
source = msm.where_is_attribute(composite_system, 'coordinates')
print(f"Coordinates are coming from: {source}")

### 3. Transparent Access and Data Geometry

You can use `get()` exactly as if it were a single file. MolSysMT handles the internal routing automatically. 

One crucial rule to remember: in MolSysMT, **coordinates are always returned as a 3D array** with the shape:  
`[n_structures, n_atoms, 3]` (where 3 represents the X, Y, Z components).

This consistency is vital. Even if you have only one structure or one atom, the array will keep its three dimensions to make your analysis scripts robust.

In [ ]:
# Get data from both sources in a single call. 
# We select the first 5 atoms to demonstrate the coordinate array shape.
names, coords = msm.get(composite_system, selection=[0, 1, 2, 3, 4], atom_name=True, coordinates=True)

print(f"Atom names: {names}")
print(f"Coordinates shape: {coords.shape} (n_structures, n_atoms, spatial:x,y,z)")

# Accessing a very specific value: 
# Y coordinate (index 1) of the 3rd atom (index 2) in the 4th structure (index 3)
print(f"Y coordinate of the 3rd atom in the 4th structure: {coords[3, 2, 1]}")

--- 

### 🏆 Challenge 3: The System Weaver

1. Take a PDB ID (e.g., `'pdb_id:181L'`) and an empty native object (`msm.native.MolSys()`). 
2. Pass them as a list to `msm.get_attributes()`.
3. Swap the order: `[msm.native.MolSys(), 'pdb_id:181L']`. Is the system still functional?
4. Visualize the composite system of Villin but showing only the protein atoms.

Congratulations! You have finished **Phase 1: Explorer**. You can now load, see, audit, and combine any molecular data. 

In **Module 4**, we will learn to visualize these systems with ease.

```{key-takeaway}
MolSysMT allows you to virtually merge multiple data sources into a single system by simply passing them as a list, with the first source taking precedence for overlapping attributes.
```